# Projet final en fondement de l'IA - Ultimate Tic Tac Toe

Ce notebook implémente un IA pour jouer à l'Ultimate Tic Tac Toe. On utilise une IA de type Minimax avec élégage alpha-beta. Les règles du jeu sont expliqué dans le rapport donné avec ce notebook. La démarche logique est expliqué dans ce notebook mais des explications complémentaires ou formulées autrement sont disponible dans le rapport.

## Representation du plateau (cf rapport)

- `board[r][c]` = valeur à la case (ligne r, colonne c), indices 0 à 8
- Valeurs : `0` = vide, `1` = joueur X (croix), `2` = joueur O (rond)
- Le grand morpion (méta-grille) : `meta[i][j]` pour le bloc (i,j), `3` = nul
- Convention d'affichage : colonnes/lignes numérotées de **1 à 9, colonne d'abord puis ligne


*Groupe: Azel OUABDESSELAM - Emilie NOE - Margaux PAOLI - Thiya PAPUGE*

## 1. Structure du plateau et imports

In [ ]:
import math
import time
import copy
import sys

def create_board():
    return [[0] * 9 for _ in range(9)]


def create_meta():
    return [[0] * 3 for _ in range(3)]

## 2. Affichage du plateau

In [ ]:
def display_board(board, meta, active_block=None):
    symbols = {0: '.', 1: 'X', 2: 'O'}

    print()
    # En-tête des colonnes
    print("   1 2 3 | 4 5 6 | 7 8 9")

    for i in range(9):
        bi = i // 3
        row_str = f"{i+1:2d} "

        for j in range(9):
            bj = j // 3

            if j in [3, 6]:
                row_str += "| "

            cell = board[i][j]

            # Si le bloc est terminé, on affiche '#' pour les cases
            if meta[bi][bj] != 0:
                row_str += "# "
            else:
                row_str += symbols[cell] + " "

        print(row_str)

        # Séparateur horizontal entre blocs
        if i in [2, 5]:
            print("   ---------------------")

    # On aff de la méta-grille
    print()
    print("=== MÉTA-GRILLE (grands morpions) ===")
    meta_sym = {0: '.', 1: 'X', 2: 'O', 3: '='}
    for i in range(3):
        print(" ".join(meta_sym[meta[i][j]] for j in range(3)))

    if active_block:
        print(f"\nBloc actif : ligne {active_block[0]+1}, colonne {active_block[1]+1}")
    else:
        print("\nTous les blocs sont jouables.")


# Test d'affichage rapide
board_test = create_board()
meta_test  = create_meta()
board_test[0][0] = 1  # X en haut à gauche
board_test[4][4] = 2  # O au centre
display_board(board_test, meta_test, active_block=(1, 1))

## 3. Fonctions utilitaires (victoire, coups legaux, application de coups)

In [ ]:
# Les 8 alignements possibles dans un morpion 3x3
WIN_LINES = [
    [0, 1, 2], [3, 4, 5], [6, 7, 8],  # lignes
    [0, 3, 6], [1, 4, 7], [2, 5, 8],  # colonnes
    [0, 4, 8], [2, 4, 6]               # diagonales
]


def check_small_board(board, bi, bj):
    # Extraire les 9 cases du petit morpion
    r0, c0 = 3 * bi, 3 * bj
    cells = []
    for dr in range(3):
        for dc in range(3):
            cells.append(board[r0 + dr][c0 + dc])

    # Vérifier chaque alignement gagnant
    for line in WIN_LINES:
        a, b, c = cells[line[0]], cells[line[1]], cells[line[2]]
        if a != 0 and a == b == c:
            return a  # 1 ou 2

    # Nul si toutes les cases sont remplies
    if all(c != 0 for c in cells):
        return 3

    return 0  # pas encore terminé


def check_meta(meta):
    meta_flat = [meta[i][j] for i in range(3) for j in range(3)]

    for line in WIN_LINES:
        a, b, c = meta_flat[line[0]], meta_flat[line[1]], meta_flat[line[2]]
        # Un bloc nul (3) ne peut pas former un alignement
        if a in (1, 2) and a == b == c:
            return a

    # Tous les blocs terminés -> nul
    if all(meta[i][j] != 0 for i in range(3) for j in range(3)):
        return 3

    return 0


def get_legal_moves(board, meta, active_block):
    moves = []

    if active_block is not None:
        bi, bj = active_block
        if meta[bi][bj] == 0:  # bloc cible encore jouable
            r0, c0 = 3 * bi, 3 * bj
            for dr in range(3):
                for dc in range(3):
                    r, c = r0 + dr, c0 + dc
                    if board[r][c] == 0:
                        moves.append((r, c))
            return moves

    # Pas de contrainte : tous les blocs non terminés
    for bi in range(3):
        for bj in range(3):
            if meta[bi][bj] == 0:
                r0, c0 = 3 * bi, 3 * bj
                for dr in range(3):
                    for dc in range(3):
                        r, c = r0 + dr, c0 + dc
                        if board[r][c] == 0:
                            moves.append((r, c))
    return moves


def get_next_block(r, c, meta):
    dr, dc = r % 3, c % 3
    if meta[dr][dc] == 0:
        return (dr, dc)
    return None

In [ ]:
def apply_move(board, meta, r, c, player):
    board[r][c] = player

    bi, bj = r // 3, c // 3
    old_meta = meta[bi][bj]

    # Mettre à jour la méta si le bloc vient d'être gagné/complété
    if old_meta == 0:
        result = check_small_board(board, bi, bj)
        if result != 0:
            meta[bi][bj] = result

    return (r, c, old_meta)


def undo_move(board, meta, undo_info):
    """
    Annule un coup précédemment appliqué par apply_move.
    """
    r, c, old_meta = undo_info
    board[r][c] = 0
    bi, bj = r // 3, c // 3
    meta[bi][bj] = old_meta


def apply_move_copy(board, meta, r, c, player):
    new_board = [row[:] for row in board]
    new_meta  = [row[:] for row in meta]
    new_board[r][c] = player

    bi, bj = r // 3, c // 3
    if new_meta[bi][bj] == 0:
        result = check_small_board(new_board, bi, bj)
        if result != 0:
            new_meta[bi][bj] = result

    return new_board, new_meta

In [ ]:
# On test pour vérifier que tout marche
b_test = create_board()
m_test = create_meta()
for c in range(3):
    b_test[0][c] = 1
print(f"Victoire X bloc (0,0) : {check_small_board(b_test, 0, 0)}  (attendu 1)")

moves = get_legal_moves(b_test, m_test, active_block=(0, 0))
print(f"Coups légaux bloc (0,0) : {len(moves)} cases  (attendu 6)")
print("Fonctions utilitaires ok.")

## 4. Heuristique d'evaluation

L'heuristique donne un score à un état du jeu sans explorer jusqu'aux feuilles.
Plus le score est élevé, plus l'état est favorable à l'IA. Voir le rapport pour plus de détails.

### Critères utilisés
| Critère | Poids | Justification |
|---------|-------|---------------|
| Victoire/Défaite globale | ±100 000 | Cas terminal |
| Blocs gagnés | ±200 | Avantage matériel |
| Menaces méta (2 blocs alignés) | ±900 | Menace de victoire globale |
| Menaces dans les petits morpions | ±10 | Menace de gagner un bloc |
| Positions stratégiques (centre, coins) | ±1-4 | Contrôle positionnel |
| Pénalité d'envoi dans bloc dangereux | -30 | Ne pas offrir de bloc à l'adversaire |

In [ ]:
CELL_WEIGHTS = [
    3, 2, 3,
    2, 4, 2,
    3, 2, 3
]

META_WEIGHTS = [
    3, 2, 3,
    2, 4, 2,
    3, 2, 3
]

# Cache pour les petits morpions
small_board_cache = {}

def clear_small_board_cache():
    global small_board_cache
    small_board_cache = {}

def get_small_board_key(board, bi, bj, player):
    h = (bi << 4) | (bj << 2) | player
    r0, c0 = 3 * bi, 3 * bj
    for dr in range(3):
        for dc in range(3):
            h = (h << 2) | board[r0 + dr][c0 + dc]
    return h

def score_line(vals, player):
    opponent = 3 - player
    if opponent in vals:
        return 0
    count = vals.count(player)
    if count == 2:
        return 10
    elif count == 1:
        return 1
    return 0

def evaluate_small_board(board, bi, bj, player):
    # Cache lookup
    key = get_small_board_key(board, bi, bj, player)
    if key in small_board_cache:
        return small_board_cache[key]
    
    r0, c0 = 3 * bi, 3 * bj
    cells = [board[r0 + dr][c0 + dc] for dr in range(3) for dc in range(3)]

    opponent = 3 - player
    score = 0

    for line in WIN_LINES:
        vals = [cells[k] for k in line]
        score += score_line(vals, player)
        score -= score_line(vals, opponent)

    for idx, val in enumerate(cells):
        if val == player:
            score += CELL_WEIGHTS[idx] * 0.1
        elif val == opponent:
            score -= CELL_WEIGHTS[idx] * 0.1

    small_board_cache[key] = score
    return score

def evaluate(board, meta, player):
    opponent = 3 - player
    score = 0

    result = check_meta(meta)
    if result == player:
        return 100000
    elif result == opponent:
        return -100000
    elif result == 3:
        return 0

    meta_flat = [meta[i][j] for i in range(3) for j in range(3)]

    # V3 AJUSTÉ: Meta threat ↑ (120 vs 90), block_won ↑ (250 vs 200)
    for line in WIN_LINES:
        vals = []
        for k in line:
            v = meta_flat[k]
            if v in (player, opponent):
                vals.append(v)
            else:
                vals.append(0)
        score += score_line(vals, player) * 120  # was 90
        score -= score_line(vals, opponent) * 120

    for i in range(3):
        for j in range(3):
            meta_idx = i * 3 + j
            w = META_WEIGHTS[meta_idx]
            if meta[i][j] == player:
                score += 250 * w / 4  # was 200
            elif meta[i][j] == opponent:
                score -= 250 * w / 4

    for bi in range(3):
        for bj in range(3):
            if meta[bi][bj] == 0:
                meta_idx = bi * 3 + bj
                w = META_WEIGHTS[meta_idx] / 4
                score += evaluate_small_board(board, bi, bj, player) * w

    return score

## 5. Algorithme Minimax avec Elagage Alpha-Beta

### Principe du Minimax
- L'IA simule les parties possibles jusqu'à une certaine profondeur
- MAX (notre IA) cherche à maximiser son score
- MIN (l'adversaire) cherche à minimiser le score de l'IA

### Élagage Alpha-Beta
- Alpha = meilleur score garanti pour MAX
- Beta = meilleur score garanti pour MIN
- Si `alpha ≥ beta` -> on coupe cette branche (inutile de l'explorer)

### Optimisations implémentées
1. Move ordering : trier les coups du plus prometteur au moins prometteur
   pour maximiser les coupures alpha-beta
2. Iterative deepening : augmenter progressivement la profondeur
   et s'arrêter quand le temps imparti est écoulé

In [ ]:
# Profondeur maximale de recherche par défaut
MAX_DEPTH = 5

# Temps maximum par coup en secondes
TIME_LIMIT = 9.5

eval_cache = {}

def clear_eval_cache():
    global eval_cache
    eval_cache = {}

def get_cache_key(board, meta, player):
    h = player
    for r in range(9):
        for c in range(9):
            h = h * 3 + board[r][c]
    for i in range(3):
        for j in range(3):
            h = h * 4 + meta[i][j]
    return h

def order_moves_fast(moves, board, meta, player):
    """Ordonne les coups sans JAMAIS modifier le board original."""
    opponent = 3 - player
    priorities = []

    for move in moves:
        r, c = move
        bi, bj = r // 3, c // 3
        dr, dc = r % 3, c % 3
        priority = CELL_WEIGHTS[dr * 3 + dc]

        # Copie virtuelle du seul bloc testé
        r0, c0 = 3 * bi, 3 * bj
        temp = [board[r0 + dr_t][c0 + dc_t] for dr_t in range(3) for dc_t in range(3)]

        temp[dr * 3 + dc] = player
        if any(temp[l[0]] != 0 and temp[l[0]] == temp[l[1]] == temp[l[2]] for l in WIN_LINES):
            priorities.append((10000, move))
            continue

        temp[dr * 3 + dc] = opponent
        if any(temp[l[0]] != 0 and temp[l[0]] == temp[l[1]] == temp[l[2]] for l in WIN_LINES):
            priorities.append((5000 + priority, move))
            continue

        priorities.append((priority, move))

    return [m for _, m in sorted(priorities, reverse=True)]


class TimeoutException(Exception):
    pass


def minimax(board, meta, depth, alpha, beta, is_maximizing, ai_player, active_block, deadline=None):
    if deadline and depth % 4 == 0 and time.time() > deadline:
        raise TimeoutException()

    terminal = check_meta(meta)
    if terminal != 0:
        if terminal == ai_player:
            return 100000 + depth
        elif terminal == 3 - ai_player:
            return -100000 - depth
        else:
            return 0

    if depth == 0:
        cache_key = get_cache_key(board, meta, ai_player)
        if cache_key in eval_cache:
            return eval_cache[cache_key]
        score = evaluate(board, meta, ai_player)
        eval_cache[cache_key] = score
        return score

    moves = get_legal_moves(board, meta, active_block)
    if not moves:
        return evaluate(board, meta, ai_player)

    current_player = ai_player if is_maximizing else 3 - ai_player
    moves = order_moves_fast(moves, board, meta, current_player)

    if is_maximizing:
        best_score = -math.inf
        for r, c in moves:
            undo_info = apply_move(board, meta, r, c, current_player)
            next_block = get_next_block(r, c, meta)
            try:
                score = minimax(board, meta, depth - 1, alpha, beta,
                                False, ai_player, next_block, deadline)
            finally:
                undo_move(board, meta, undo_info)  # TOUJOURS annulé

            best_score = max(best_score, score)
            alpha = max(alpha, best_score)
            if alpha >= beta:
                break
        return best_score

    else:
        best_score = math.inf
        for r, c in moves:
            undo_info = apply_move(board, meta, r, c, current_player)
            next_block = get_next_block(r, c, meta)
            try:
                score = minimax(board, meta, depth - 1, alpha, beta,
                                True, ai_player, next_block, deadline)
            finally:
                undo_move(board, meta, undo_info)  # TOUJOURS annulé

            best_score = min(best_score, score)
            beta = min(beta, best_score)
            if alpha >= beta:
                break
        return best_score


def get_best_move(board, meta, ai_player, active_block):
    clear_eval_cache()

    moves = get_legal_moves(board, meta, active_block)
    if not moves:
        return None
    if len(moves) == 1:
        print(f"[IA] Un seul coup possible : colonne {moves[0][1]+1}, ligne {moves[0][0]+1}")
        return moves[0]

    start_time = time.time()
    deadline = start_time + TIME_LIMIT

    best_move = moves[0]
    best_score = -math.inf
    reached_depth = 0

    sorted_moves = order_moves_fast(moves, board, meta, ai_player)

    for depth in range(1, 20):
        try:
            current_best_move = moves[0]
            current_best_score = -math.inf

            for r, c in sorted_moves:
                if time.time() > deadline - 0.1:
                    raise TimeoutException()

                undo_info = apply_move(board, meta, r, c, ai_player)
                next_block = get_next_block(r, c, meta)
                try:
                    score = minimax(board, meta, depth - 1,
                                    -math.inf, math.inf, False,
                                    ai_player, next_block, deadline)
                finally:
                    undo_move(board, meta, undo_info)  # TOUJOURS annulé

                if score > current_best_score:
                    current_best_score = score
                    current_best_move = (r, c)

                if current_best_score >= 100000:
                    break

            best_move = current_best_move
            best_score = current_best_score
            reached_depth = depth

            if best_score >= 100000:
                break

        except TimeoutException:
            break

    elapsed = time.time() - start_time
    print(f"[IA] Coup choisi : colonne {best_move[1]+1}, ligne {best_move[0]+1} "
          f"(score={best_score:.0f}, profondeur={reached_depth}, temps={elapsed:.2f}s)")

    return best_move

## 6. Boucle de jeu principale

### Modes disponibles
| Mode | `ai_player` | `human_player` | Description |
|------|-------------|----------------|-------------|
| IA = X, Humain = O | `1` | `2` | L'IA commence |
| Humain = X, IA = O | `2` | `1` | Le joueur commence |
| IA vs IA (relai) | `1` | `None` | Saisir les coups de l'autre IA manuellement |

In [ ]:
# BOUCLE DE JEU PRINCIPALE

def get_human_move(board, meta, active_block, player):
    legal = get_legal_moves(board, meta, active_block)
    symbols = {0: "vide", 1: "X", 2: "O"}

    while True:
        try:
            if active_block:
                bi, bj = active_block
                col_min, col_max = bj * 3 + 1, bj * 3 + 3
                row_min, row_max = bi * 3 + 1, bi * 3 + 3
                hint = f" (bloc: col {col_min}-{col_max}, lig {row_min}-{row_max})"
            else:
                hint = " (tous blocs jouables)"

            sys.stdout.flush()  # Garantit que tous les prints précédents sont affichés avant input
            raw = input(f"Joueur {'X' if player == 1 else 'O'} -- col,lig{hint} : ")

            parts = raw.replace(' ', '').split(',')
            if len(parts) != 2:
                print("Format invalide, entre ton coup au format col,lig.")
                print("   Exemple : 5,3 pour colonne 5, ligne 3.")
                continue

            col, row = int(parts[0]), int(parts[1])

            if not (1 <= col <= 9):
                print(f"Colonne {col} hors limites, elle doit etre entre 1 et 9.")
                continue
            if not (1 <= row <= 9):
                print(f"Ligne {row} hors limites, elle doit etre entre 1 et 9.")
                continue

            c, r = col - 1, row - 1
            bi, bj = r // 3, c // 3

            if (r, c) in legal:
                print(f"   OK : colonne {col}, ligne {row} (bloc [{bi+1},{bj+1}]).")
                return (r, c)

            if board[r][c] != 0:
                occupant = symbols[board[r][c]]
                print(f"Case (col {col}, lig {row}) deja occupee par {occupant}.")
                continue

            meta_val = meta[bi][bj]
            if meta_val != 0:
                status = {1: "gagne par X", 2: "gagne par O", 3: "nul (termine)"}
                print(f"Le bloc [{bi+1},{bj+1}] est deja {status.get(meta_val, 'termine')}.")
                print("   Tu ne peux plus jouer dans un bloc termine.")
                continue

            if active_block:
                abi, abj = active_block
                print(f"Mauvais bloc, tu vises le bloc [{bi+1},{bj+1}] "
                      f"mais tu dois jouer dans le bloc [{abi+1},{abj+1}].")
                col_min, col_max = abj * 3 + 1, abj * 3 + 3
                row_min, row_max = abi * 3 + 1, abi * 3 + 3
                print(f"   Colonnes {col_min}-{col_max}, lignes {row_min}-{row_max}")
                continue

            print("Coup illegal, essaie une autre case.")

        except ValueError:
            print("Saisie invalide, entre deux nombres separes par une virgule.")
            print("   Exemple : 5,3 pour colonne 5, ligne 3.")


def play_game(ai_player=1, human_player=2):
    board = create_board()
    meta  = create_meta()
    current_player = 1
    active_block   = None

    symbols = {1: 'X', 2: 'O'}
    times = {1: 0.0, 2: 0.0}

    print("\n" + "=" * 50)
    print("ULTIMATE TIC TAC TOE -- Debut de partie")
    print(f"IA joue : {symbols[ai_player]}")
    if human_player:
        print(f"Humain joue : {symbols[human_player]}")
    else:
        print("Mode : IA vs IA (entrez les coups de l'autre IA manuellement)")
    print("=" * 50)

    display_board(board, meta, active_block)

    move_count = 0

    while True:
        print(f"\n--- Tour du joueur {symbols[current_player]} (tour {move_count + 1}) ---")

        if current_player == ai_player:
            print("[IA réfléchit...]", flush=True)
            t0 = time.time()
            move = get_best_move(board, meta, ai_player, active_block)
            elapsed = time.time() - t0
            times[ai_player] += elapsed
            clear_small_board_cache()
        else:
            t0 = time.time()
            move = get_human_move(board, meta, active_block, current_player)
            elapsed = time.time() - t0
            times[current_player] += elapsed

        if move is None:
            print("Aucun coup possible, partie nulle.")
            break

        r, c = move
        move_count += 1

        apply_move(board, meta, r, c, current_player)
        active_block = get_next_block(r, c, meta)

        print(f"{symbols[current_player]} joue : colonne {c+1}, ligne {r+1}", flush=True)
        display_board(board, meta, active_block)
        sys.stdout.flush()

        result = check_meta(meta)
        if result != 0:
            print("\n" + "=" * 50)
            if result == 1:
                print("X (croix) a gagne la partie !")
            elif result == 2:
                print("O (ronds) a gagne la partie !")
            else:
                x_blocks = sum(meta[i][j] == 1 for i in range(3) for j in range(3))
                o_blocks = sum(meta[i][j] == 2 for i in range(3) for j in range(3))
                print(f"Match nul ! X : {x_blocks} blocs, O : {o_blocks} blocs")
                if x_blocks > o_blocks:
                    print("X gagne au nombre de blocs.")
                elif o_blocks > x_blocks:
                    print("O gagne au nombre de blocs.")
                else:
                    print("Egalite parfaite.")

            print(f"\nTemps total X : {times[1]:.2f}s")
            print(f"Temps total O : {times[2]:.2f}s")
            print("=" * 50)
            break

        current_player = 3 - current_player

## 7. Lancer une partie

### Configuration
- `MAX_DEPTH` : profondeur maximale (défaut 5, recommandé 4-7)
- `TIME_LIMIT` : temps maximum par coup en secondes (défaut 10s)

### Modes de jeu
- `play_game(ai_player=1, human_player=2)` -> IA joue X, humain joue O
- `play_game(ai_player=2, human_player=1)` -> humain joue X, IA joue O
- `play_game(ai_player=1, human_player=None)` -> IA vs IA via humains interposés

In [ ]:
# Réglages de l'IA
MAX_DEPTH = 5       # profondeur max (augmenter si la machine le permet)
TIME_LIMIT = 5    # secondes max par coup

# Test IA vs IA automatique
def play_game_auto(verbose=False):
    """IA vs IA entièrement automatique pour évaluer la force."""
    board = create_board()
    meta = create_meta()
    current_player = 1
    active_block = None
    
    symbols = {1: 'X', 2: 'O'}
    times = {1: 0.0, 2: 0.0}
    move_count = 0
    
    if verbose:
        print("\n" + "=" * 50)
        print("ULTIMATE TIC TAC TOE -- IA vs IA Auto")
        print("=" * 50)
        display_board(board, meta, active_block)
    
    while True:
        if verbose:
            print(f"\n--- {symbols[current_player]} tour {move_count + 1} ---")
        
        t0 = time.time()
        move = get_best_move(board, meta, current_player, active_block)
        elapsed = time.time() - t0
        times[current_player] += elapsed
        
        if move is None:
            return (3, times, move_count)  # nul
        
        r, c = move
        move_count += 1
        
        apply_move(board, meta, r, c, current_player)
        active_block = get_next_block(r, c, meta)
        
        if verbose:
            print(f"{symbols[current_player]} → col {c+1}, lig {r+1} ({elapsed:.2f}s)")
            display_board(board, meta, active_block)
        
        clear_small_board_cache()
        
        result = check_meta(meta)
        if result != 0:
            if verbose:
                print("\n" + "=" * 50)
                if result == 1:
                    print("✓ X GAGNE!")
                elif result == 2:
                    print("✓ O GAGNE!")
                else:
                    x_blocks = sum(meta[i][j] == 1 for i in range(3) for j in range(3))
                    o_blocks = sum(meta[i][j] == 2 for i in range(3) for j in range(3))
                    print(f"NUL - X: {x_blocks} blocs, O: {o_blocks} blocs")
                print(f"Temps X: {times[1]:.2f}s, Temps O: {times[2]:.2f}s")
                print("=" * 50)
            return (result, times, move_count)
        
        current_player = 3 - current_player

# Pour jouer:
play_game(ai_player=1, human_player=2)  # IA joue X
# play_game(ai_player=2, human_player=1)  # IA joue O
# play_game_auto(verbose=True)            # IA vs IA auto